---

NDS-Mini

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl
from pgl.pglImage import pglImageDatabase, pglImage
from pgl.pglMessages import pglMessages
from pgl.pglExperiment import pglTask, pglExperiment
from pgl.pglParameter import pglParameter, pglParameterBatch
from pgl.pglImage import pglImageFileNSDMini
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

pgl = pgl()

# close any existing windows
pgl.cleanUp()
nsdminiStimuliFilePath = "/Users/justin/proj/nsdmini/nsdmini_stimuli.hdf5"

IndentationError: unindent does not match any outer indentation level (pglImage.py, line 1116)

---

NSD-mini task

---

In [ ]:
filePath = "/Users/justin/proj/nsdmini/nsdmini_stimuli.hdf5"

def expectError(errorClass, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except errorClass:
        return
    raise AssertionError(f"Expected {errorClass.__name__}")

with pglImageDatabase(
    dataPath=filePath,
    stimulusFileClass=pglImageFileNSDMini
) as database:

    database.setTrialOrder([9, -1, 2, 9])

    # Tracking must be explicitly started.
    expectError(RuntimeError, database.nextTrial)
    expectError(RuntimeError, database.completeTrial, 0)
    database.startRun()

    # Retrieval does not advance or load pixels.
    firstTrial = database.nextTrial()
    assert firstTrial is database.getTrial(0)
    assert database.nextTrial() is firstTrial
    assert all(stimulus._img is None for stimulus in database.stimuli)
    print("Explicit start and non-advancing retrieval: passed")

    # Reject mistakes without changing the current trial.
    expectError(RuntimeError, database.startRun)
    expectError(RuntimeError, database.completeTrial, 2)
    expectError(ValueError, database.completeTrial, 999)
    expectError(TypeError, database.completeTrial, True)
    expectError(RuntimeError, database.setTrialOrder, [2, 9])
    assert database.nextTrial() is firstTrial
    print("Run-state and completion validation: passed")

    # Completing the first trial advances to the blank.
    assert database.completeTrial(firstTrial.trialId) is firstTrial
    assert firstTrial.status == "completed"
    assert firstTrial.completedAt is not None
    expectError(RuntimeError, database.completeTrial, firstTrial.trialId)

    blankTrial = database.nextTrial()
    assert blankTrial.trialNum == 1
    assert blankTrial.isBlank
    assert blankTrial.img is None
    database.completeTrial(blankTrial.trialId)
    assert database.nextTrial().trialNum == 2
    print("Completion, duplicate protection, and blanks: passed")

    # Finish the remaining presentations, including the repeated image.
    for trialNum in (2, 3):
        trial = database.nextTrial()
        assert trial.trialNum == trialNum
        database.completeTrial(trial.trialId)

    assert database.nextTrial() is None
    assert all(trial.status == "completed" for trial in database.trials)
    assert all(trial.completedAt is not None for trial in database.trials)
    assert all(stimulus._img is None for stimulus in database.stimuli)
    print("End-of-run behavior without image loading: passed")

    # Explicit reset clears progress, not ordering or image inventory.
    database.startRun(reset=True)
    assert database.nextTrial() is firstTrial
    assert all(trial.status == "pending" for trial in database.trials)
    assert all(trial.completedAt is None for trial in database.trials)
    print("Explicit run reset: passed")

In [ ]:
class pglNSDMiniTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "NSD-mini"
        #self.settings.nTrials = 23
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.config = {
            
        }
        
        # setup digitalIO
        from pgl import pglLabJack
        self.digIO = pglLabJack()
        #from pgl import pglDataPixx
        #self.dataPixx = pglDataPixx()

        if self.digIO.isActive:
            try:
                # initialize digital ports
                for channel in range(8):
                    self.digIO.setupDigitalOutput(channel=channel, group="FI0", pulseLen=3)
                self.digIO.setupDigitalOutputWord(channels=[0,1,2,3,4,5,6,7])
                #self.digIO.setupConditions(numBits=p['numConditionBits'])
                # keep a pointer for digIO (so in principle the code can work with a differnt digIO device)
                #self.digIO = self.dataPixx
                # add to pgl, so that pgl.poll() will return button press events
                #self.pgl.devicesAdd(self.digIO)
            except Exception as e:
                pglMessages.warning(f"Error setting up dataPixx: {e}")    
                #self.digIO = None
                self.dataPixx = None        
        else:
            pglMessages.warning("DataPixx not active, digitalIO and button presses will not function")
            #self.digIO = None
            self.dataPixx = None
        

        # set seglens, 
        # 1st segment is image display
        # 2nd segment is blank
        self.settings.seglen = [0.5, 0.5] * p['nImagesPerTrial']

        # initialize image database using parameters set from fixedParameters
        self.state._imdb = pglImageDatabase(p['imagesDirectory'])
        if self.state._imdb.nStimuli==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        # load manifest
        self.state._imdb.useManifest(
            filenameColumn=p['manifestColumnNames']['filenameColumn'],
            indexColumn=p['manifestColumnNames']['indexColumn'],
            captionColumn=p['manifestColumnNames']['captionColumn'],            
        )
        
        # initialize catch image database using parameters set from fixedParameters
        self.state._imdbCatch = pglImageDatabase(p['catchImagesDirectory'])
        if self.state._imdbCatch.nStimuli==0:
            pglMessages.warning(f"No images found in {p['imagesDirectory']}")
            return
        # load manifest
        self.state._imdbCatch.useManifest(
            filenameColumn=p['catchManifestColumnNames']['filenameColumn'],
            indexColumn=p['catchManifestColumnNames']['indexColumn'],
            captionColumn=p['catchManifestColumnNames']['captionColumn'],
        )
        
        # preload images
        for iImage in range(p['nImages']):
            self.state._imdb.preload(iImage)
        for iImage in range(p['nCatchImages']):
            self.state._imdbCatch.preload(iImage)
            
        # add parameter for image number
        imageNumParameter = pglParameterBatch('imageNum',np.arange(p['nImages']),batchSize=p['nImagesPerTrial'], catchTrialEvery=p['catchTrialEvery'], catchBatchEvery=p['catchBatchEvery'])
        self.addParameter(imageNumParameter)
        
        # create a self-tracked parameter of the catch images
        self.state.catchNumParameter = pglParameter('catchNum',np.arange(p['nCatchImages']))
        
        # set current values
        self.state.currentImage = None
        self.state.isCatch= False
        self.state.wasCatch = False
        self.state.catchConditionNum = p['catchConditionNum']
        self.state.catchBatchConditionNum = p['catchBatchConditionNum']

        imageNumParameter.print()
    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment % 2 == 0: 
            # remember if the last segment was a catch
            self.state.wasCatch = self.state.isCatch
            self.state.gotResponse = False
            # set fixation colro
            self.state.fixColor = 1
            # get the current image number
            imageNums = self.currentParams['imageNum']
            if imageNums:
                # load an image
                imageNum = imageNums[int(self.state.currentSegment/2)]
                if imageNum:
                    # get the image data
                    img = self.state._imdb.get(imageNum)
                    self.state.isCatch = False
                else:
                    # get a catch image
                    catchNum = self.state.catchNumParameter.get()['catchNum']
                    img = self.state._imdbCatch.get(catchNum)    
                    self.state.isCatch = True
                    imageNums[int(self.state.currentSegment/2)] = -catchNum
                    print("CATCH: {self.state.currentSegment}")
                img.convert("RGB")
                print(f"img: {img}")
                # turn into a pglImage
                self.state._currentImage = self.pgl.imageCreate(np.array(img))
                # send digital pulse
                if self.digIO is not None: 
                    #self.dataPixx.writeCondition(imageNum)
                    if self.state.isCatch:
                        #self.dataPixx.writeCondition(self.state.catchConditionNum)
                        self.digIO.digitalOutputWord(self.state.catchConditionNum)
                    else:
                        #self.dataPixx.writeCondition(int(imageNum)+1)
                        self.digIO.digitalOutputWord(int(imageNum)+1)
            else:
                # catch batch
                self.state._currentImage = None                 
                #self.dataPixx.writeCondition(self.state.catchBatchConditionNum)
                self.digIO.digitalOutputWord(self.state.catchBatchConditionNum)

    ########################
    # handleSubjectResponse
    ########################    
    def handleSubjectResponse(self, response, updateTime):
        '''
        Handle the subject response. Response will come in as an integer
        value of what button was pressed. The order of buttons is set
        in pgl.settings() in the field "responseKeys"
        '''
        # already received a response
        if self.state.gotResponse: return None
        # mark that we got a response
        self.state.gotResponse = True
        
        # if this was a catchTrial or the last one was, then it is correct
        if self.state.isCatch or self.state.wasCatch:
            correct = True 
            self.state.fixColor = [0,1,0]
        else:
            correct = False
            self.state.fixColor = [1,0,0]
            
        # return response type
        return correct

    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment % 2 == 0: 
            if self.state._currentImage:
                self.state._currentImage.display(height=self.settings.fixedParameters['imageSize'])
        
        # Draw ABC fixation cross from Thaler, Schütz, Goodale & Gegenfurtner (2013) Vision Research 76:31-42
        pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
        pgl.rect(-0.3,0,width=0.6,height=0.15,color=self.state.fixColor,hAlign='left',vAlign='center')
        pgl.rect(0,-0.3,width=0.15,height=0.6,color=self.state.fixColor,hAlign='center',vAlign='top')
        pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

In [ ]:
import h5py

def inspectItem(name, item):
    if isinstance(item, h5py.Dataset):
        print(f"Dataset: {name} | shape={item.shape} | dtype={item.dtype}")
    else:
        print(f"Group:   {name}")

nsdminiStimuliFilePath = "/Users/justin/proj/nsdmini/nsdmini_stimuli.hdf5"
with h5py.File(nsdminiStimuliFilePath, "r") as hdfFile:
    # Show items at the root of the file.
    print("Top-level items:", list(hdfFile.keys()))

    # Walk through every group and dataset without loading the array data.
    hdfFile.visititems(inspectItem)

    # Show metadata attached to the file itself.
    print("\nFile attributes:", dict(hdfFile.attrs))

In [ ]:
import h5py
import matplotlib.pyplot as plt

def loadImage(filePath, imageIndex):
    with h5py.File(filePath, "r") as hdfFile:
        dataset = hdfFile["imgBrick"]

        # Image indices are zero-based: 0 through 514.
        if not 0 <= imageIndex < dataset.shape[0]:
            raise IndexError(f"imageIndex must be between 0 and {dataset.shape[0] - 1}")

        # Load only the selected image into memory.
        return dataset[imageIndex]

# Select an image: index 9 is the 10th image.
imageIndex =125
image = loadImage(nsdminiStimuliFilePath, imageIndex)
    
print(f"Image shape: {image.shape}, dtype: {image.dtype}")

plt.imshow(image)
plt.title(f"Image {imageIndex + 1}")
plt.axis("off")
plt.show()

In [ ]:
import scipy.io as sio

matData = sio.loadmat("/Users/justin/proj/nsdmini/nsdmini_expdesign.mat", simplify_cells=True)

# Show user variables, excluding MATLAB metadata.
for name, value in matData.items():
    if not name.startswith("__"):
        print(f"{name}: type={type(value).__name__}, shape={getattr(value, 'shape', None)}")

# Access a variable by its name.
data = matData["masterordering"]

In [ ]:
print(data)